<h1>Chapter 9 - Graph RAG: Recipe 4: Embeddings & Vector Search</h1>
<i>Building knowledge graphs and using them in RAG systems.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch09_graph_rag/9.4_embeddings_vector_search.ipynb)

---

This notebook is for Chapter 9 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [ ]:
%pip install python-dotenv==1.0.0 neo4j==5.0.0 openai==1.12.0 pandas==2.0.0

## 1. Setup and Imports

Import required libraries and load environment variables.

In [ ]:
from __future__ import annotations
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()
client = OpenAI()

## 2. Connect to Neo4j

Create a reusable driver function that reads connection details from environment variables.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
OPENAI_API_KEY=your_openai_api_key
```

In [ ]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    pwd = os.getenv("NEO4J_PASSWORD")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Test connection
driver = get_driver()
print("✓ Connected to Neo4j")
driver.close()

## 3. Create Embeddings for Clauses

Generate embeddings for all clause text using OpenAI's text-embedding-3-small model and store them in the graph.

In [ ]:
def create_embedding(text: str):
    """Generate embeddings using OpenAI's text-embedding-3-small model."""
    return (
        client.embeddings.create(model="text-embedding-3-small", input=text)
        .data[0]
        .embedding
    )

def create_clause_embeddings():
    """Add embeddings to all Clause nodes in the graph."""
    driver = get_driver()
    with driver.session() as session:
        result = session.run("MATCH (cl:Clause) RETURN cl.id AS id, cl.text AS text")
        clauses = list(result)
        print(f"Creating embeddings for {len(clauses)} clauses...")
        
        for i, row in enumerate(clauses, 1):
            emb = create_embedding(row["text"])
            session.run(
                "MATCH (cl:Clause {id:$id}) SET cl.embedding=$emb",
                id=row["id"],
                emb=emb,
            )
            if i % 10 == 0:
                print(f"  Processed {i}/{len(clauses)} clauses")
    driver.close()
    print("✓ All clause embeddings created")

# Execute - This may take a few minutes depending on the number of clauses
create_clause_embeddings()

## 4. Create Vector Index

Create a vector index for efficient approximate nearest neighbor (ANN) search on clause embeddings.

In [ ]:
def create_vector_index():
    """Create a vector index for semantic search on Clause embeddings."""
    driver = get_driver()
    with driver.session() as session:
        session.run(
            """
            CREATE VECTOR INDEX clause_embeddings IF NOT EXISTS
            FOR (c:Clause) ON c.embedding
            OPTIONS { 
                indexConfig: { 
                    `vector.dimensions`: 1536, 
                    `vector.similarity_function`: "cosine" 
                } 
            }
            """
        )
    driver.close()
    print("✓ Vector index created")

# Execute
create_vector_index()

## 5. Semantic Search

Find clauses semantically similar to a natural language query using vector similarity search.

In [ ]:
def semantic_search(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Find clauses semantically similar to the query."""
    driver = get_driver()
    emb = create_embedding(query)
    cypher = """
    CALL db.index.vector.queryNodes(
        "clause_embeddings", $top_k, $embedding
    )
    YIELD node, score
    RETURN node.title AS title, 
           node.text AS text, 
           score 
    ORDER BY score DESC
    """
    with driver.session() as session:
        rows = [r.data() for r in session.run(cypher, top_k=top_k, embedding=emb)]
    driver.close()
    return rows

# Execute
query = "What is the uptime guarantee?"
results = semantic_search(query, top_k=3)
print(f"✓ Found {len(results)} semantically similar clauses for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['title']} (score: {result['score']:.4f})")
    text_preview = result['text'][:150] + "..." if len(result['text']) > 150 else result['text']
    print(f"   {text_preview}\n")

## 6. Hybrid Search by Industry

Combine semantic similarity search with graph-based filtering to find relevant clauses within a specific industry.

In [ ]:
def hybrid_search_by_industry(
    query: str, industry: str, top_k: int = 5
) -> List[Dict[str, Any]]:
    """Combine semantic search with industry filtering."""
    driver = get_driver()
    emb = create_embedding(query)
    cypher = """
    CALL db.index.vector.queryNodes(
        "clause_embeddings", $top_k, $embedding
    ) 
    YIELD node, score
    MATCH (node)<-[:HAS_CLAUSE]-(s:SLA)<-[:HAS_SLA]-(c:Company)
    WHERE c.industry = $industry
    RETURN c.name AS company, 
           s.id AS sla_id, 
           node.title AS clause_title, 
           node.text AS clause_text, 
           score
    ORDER BY score DESC
    """
    with driver.session() as session:
        rows = [
            r.data()
            for r in session.run(cypher, top_k=top_k, embedding=emb, industry=industry)
        ]
    driver.close()
    return rows

# Execute
query = "What are the data privacy obligations?"
industry = "Healthcare"
results = hybrid_search_by_industry(query=query, industry=industry, top_k=3)
print(f"✓ Found {len(results)} relevant clauses in {industry} industry for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['company']} - {result['clause_title']} (score: {result['score']:.4f})")
    text_preview = result['clause_text'][:150] + "..." if len(result['clause_text']) > 150 else result['clause_text']
    print(f"   {text_preview}\n")